In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [5]:
# 데이터 불러오기
cust_df = pd.read_csv("../세미파이널/data/train.csv", encoding='latin-1')
cust_df.head()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [6]:
# 원본 데이터 보존
df_clean = cust_df.copy()

# 상수 변수 제거
constant_cols = df_clean.columns[df_clean.nunique() == 1]

df_clean = df_clean.drop(columns=constant_cols)

print("제거 전 데이터 크기:", cust_df.shape)
print("제거 후 데이터 크기:", df_clean.shape)

제거 전 데이터 크기: (76020, 371)
제거 후 데이터 크기: (76020, 337)


In [7]:
# ID 변수 제거
df_clean = df_clean.drop(columns="ID")

print(df_clean.shape)

(76020, 336)


In [9]:
# var3인 고객은 불만족 고객 비율이 낮다.
# 삭제하면 정보 손실이 발생할 수 있기에 NaN 처리 
df_clean["var3"] = df_clean["var3"].replace(-999999, np.nan)

In [10]:
# NaN 개수
df_clean["var3"].isnull().sum()

np.int64(116)

In [11]:
# var3 결측이라는 새 변수 생성
df_clean["var3_missing"] = df_clean["var3"].isna().astype(int)

In [12]:
# var3 결측처리한 값 중앙값 대체
df_clean["var3"] = df_clean["var3"].fillna(
    df_clean["var3"].median()
)

In [13]:
# 희소변수를 찾는 코드
feature_cols = df_clean.drop(columns="TARGET").columns

zero_ratio = (df_clean[feature_cols] == 0).mean().sort_values(ascending=False)

zero_ratio.head(20)

saldo_medio_var29_hace3          0.999987
num_reemb_var33_ult1             0.999987
delta_num_trasp_var33_out_1y3    0.999987
delta_imp_trasp_var33_out_1y3    0.999987
delta_imp_reemb_var33_1y3        0.999987
delta_num_reemb_var33_1y3        0.999987
imp_trasp_var33_out_ult1         0.999987
imp_reemb_var33_ult1             0.999987
imp_reemb_var17_hace3            0.999987
num_reemb_var17_hace3            0.999987
num_trasp_var33_out_ult1         0.999987
saldo_medio_var13_medio_ult3     0.999974
num_var18                        0.999974
saldo_medio_var13_medio_hace2    0.999974
num_var18_0                      0.999974
ind_var34                        0.999974
saldo_var13_medio                0.999974
ind_var6                         0.999974
num_meses_var13_medio_ult3       0.999974
saldo_medio_var13_medio_ult1     0.999974
dtype: float64

In [14]:
nunique = df_clean.drop(columns="TARGET").nunique()

print("고유값 2개:", (nunique == 2).sum())
print("고유값 3~10개:", ((nunique >= 3) & (nunique <= 10)).sum())
print("고유값 11~100개:", ((nunique >= 11) & (nunique <= 100)).sum())
print("고유값 101개 이상:", (nunique > 100).sum())

고유값 2개: 106
고유값 3~10개: 99
고유값 11~100개: 66
고유값 101개 이상: 65


In [15]:
nunique = df_clean.drop(columns="TARGET").nunique()

binary_cols = nunique[nunique == 2].index.tolist()

print("이진 변수 개수:", len(binary_cols))

이진 변수 개수: 106


In [16]:
binary_eda = []

for col in binary_cols:
    temp = df_clean.groupby(col)["TARGET"].agg(
        count="count",
        dissatisfied_rate="mean"
    ).reset_index()

    for _, row in temp.iterrows():
        binary_eda.append({
            "variable": col,
            "value": row[col],
            "count": row["count"],
            "dissatisfied_rate": row["dissatisfied_rate"]
        })

binary_eda = pd.DataFrame(binary_eda)

In [18]:
# value 1만 추출
binary_target_rate = binary_eda[
    binary_eda["value"] == 1
].copy()

In [19]:
# 표본이 100명 이상인 변수만 비교
binary_target_rate = binary_target_rate[
    binary_target_rate["count"] >= 100
]

In [20]:
# 상대적인 위험도
# 단순 불만족률 보다 전체 평균 대비 몇 배 높은지 확인
overall_rate = df_clean["TARGET"].mean()

binary_target_rate["relative_risk"] = (
    binary_target_rate["dissatisfied_rate"] / overall_rate
)

binary_target_rate.sort_values(
    "relative_risk",
    ascending=False
).head(20)

,variable,value,count,dissatisfied_rate,relative_risk
13,ind_var8_0,1.0,2496.0,0.088942,2.247804
15,ind_var8,1.0,2174.0,0.071297,1.801865
111,ind_var39,1.0,283.0,0.070671,1.786050
107,ind_var40,1.0,283.0,0.070671,1.786050
3,ind_var1,1.0,286.0,0.069930,1.767315
59,ind_var25_cte,1.0,2009.0,0.067198,1.698259
63,ind_var26_cte,1.0,2095.0,0.066826,1.688862
67,ind_var25_0,1.0,1797.0,0.063996,1.617334
69,ind_var25,1.0,1797.0,0.063996,1.617334
61,ind_var26_0,1.0,1873.0,0.063001,1.592188


In [21]:
#  차이가 있는지 검정
# 카이제곱 검정
from scipy.stats import chi2_contingency

table = pd.crosstab(
    df_clean["ind_var8_0"],
    df_clean["TARGET"]
)

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p_value)

Chi-square: 164.20575643311588
p-value: 1.3639963335401472e-37


In [22]:
# 귀무가설 H₀: ind_var8_0과 TARGET은 서로 독립이다.
# p-value = 1.36 × 10⁻³⁷ 이므로 귀무가설 기각
# 따라서 ind_var8_o 과 고객 만족도 사이에 통계적으로 유의한 연관성이 있다.

In [23]:
# 효과 크기 계산
from scipy.stats import chi2_contingency
import numpy as np

table = pd.crosstab(
    df_clean["ind_var8_0"],
    df_clean["TARGET"]
)

chi2, p_value, dof, expected = chi2_contingency(table)

n = table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / n)

print("Chi-square:", chi2)
print("p-value:", p_value)
print("Cramér's V:", cramers_v)

Chi-square: 164.20575643311588
p-value: 1.3639963335401472e-37
Cramér's V: 0.046476161937365854


In [24]:
# 효과크기가 작음
# 통계적으로 유의하긴 하지만 연관성의 크기 자체는 크지 않다.

In [25]:
# 다른 변수 또한 진행
from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd

binary_results = []

for col in binary_cols:
    table = pd.crosstab(df_clean[col], df_clean["TARGET"])

    chi2, p_value, dof, expected = chi2_contingency(table)

    n = table.to_numpy().sum()
    cramers_v = np.sqrt(chi2 / n)

    # 값이 1인 고객의 불만족률
    if 1 in table.index:
        count_1 = table.loc[1].sum()
        dissatisfied_1 = table.loc[1, 1] if 1 in table.columns else 0
        rate_1 = dissatisfied_1 / count_1
    else:
        count_1 = 0
        rate_1 = np.nan

    binary_results.append({
        "variable": col,
        "n_value_1": count_1,
        "dissatisfied_rate": rate_1,
        "chi2": chi2,
        "p_value": p_value,
        "cramers_v": cramers_v
    })

binary_stats = pd.DataFrame(binary_results)

In [26]:
# 효과크기가 큰 변수부터 확인
binary_stats.sort_values(
    "cramers_v",
    ascending=False
).head(20)

,variable,n_value_1,dissatisfied_rate,chi2,p_value,cramers_v
38,ind_var30,55710,0.021935,1704.395117,0.000000e+00,0.149734
3,ind_var5,50459,0.020789,1391.157614,1.753487e-304,0.135277
6,ind_var8_0,2496,0.088942,164.205756,1.363996e-37,0.046476
17,ind_var13,3866,0.006208,118.360263,1.445898e-27,0.039458
10,ind_var13_0,3972,0.006798,117.526025,2.201918e-27,0.039319
8,ind_var12_0,5133,0.011884,110.238324,8.689042e-26,0.038080
51,ind_var39_0,66955,0.037055,92.811706,5.751395e-22,0.034941
12,ind_var13_corto,3153,0.007295,89.274820,3.436048e-21,0.034269
11,ind_var13_corto_0,3264,0.007966,88.761538,4.453920e-21,0.034170
9,ind_var12,3456,0.008970,88.359110,5.458762e-21,0.034093


In [27]:
for col in ["ind_var30", "ind_var5", "ind_var8_0"]:
    print(f"\n===== {col} =====")
    
    result = pd.crosstab(
        df_clean[col],
        df_clean["TARGET"],
        normalize="index"
    ) * 100
    
    print(result)


===== ind_var30 =====
TARGET             0         1
ind_var30                     
0          91.206302  8.793698
1          97.806498  2.193502

===== ind_var5 =====
TARGET            0         1
ind_var5                     
0         92.335981  7.664019
1         97.921084  2.078916

===== ind_var8_0 =====
TARGET              0         1
ind_var8_0                     
0           96.210761  3.789239
1           91.105769  8.894231


In [28]:
# 연속형 변수 처리 
# 1. 연속형 변수 선정
feature_cols = df_clean.drop(columns="TARGET").columns

nunique = df_clean[feature_cols].nunique()

continuous_cols = nunique[nunique > 10].index.tolist()

print("연속형 변수 후보:", len(continuous_cols))

연속형 변수 후보: 131


In [29]:
# Mann-Whitney U 검정
from scipy.stats import mannwhitneyu
import numpy as np
import pandas as pd

continuous_results = []

for col in continuous_cols:
    
    group0 = df_clean.loc[df_clean["TARGET"] == 0, col].dropna()
    group1 = df_clean.loc[df_clean["TARGET"] == 1, col].dropna()
    
    # 두 그룹 중 하나가 값이 없거나 동일한 경우 제외
    if len(group0) == 0 or len(group1) == 0:
        continue
    
    u_stat, p_value = mannwhitneyu(
        group0,
        group1,
        alternative="two-sided"
    )
    
    # Rank-biserial correlation
    n0 = len(group0)
    n1 = len(group1)
    
    rank_biserial = 1 - (2 * u_stat) / (n0 * n1)
    
    continuous_results.append({
        "variable": col,
        "median_target0": group0.median(),
        "median_target1": group1.median(),
        "mean_target0": group0.mean(),
        "mean_target1": group1.mean(),
        "p_value": p_value,
        "rank_biserial": rank_biserial
    })

continuous_stats = pd.DataFrame(continuous_results)

In [ ]:
#  효과크기가 큰 변수 확인
continuous_stats["abs_effect"] = (
    continuous_stats["rank_biserial"].abs()
)

continuous_stats.sort_values(
    "abs_effect",
    ascending=False
).head(20)

,variable,median_target0,median_target1,mean_target0,mean_target1,p_value,rank_biserial,abs_effect
1,var15,27.00,38.00,32.946406,39.680519,2.147072e-305,0.397280,0.397280
50,saldo_var30,3.00,0.00,14154.089498,2164.364721,5.672663e-286,-0.381658,0.381658
56,saldo_var42,3.00,0.00,7435.742356,1268.797769,5.370136e-245,-0.352087,0.352087
38,saldo_var5,3.00,0.00,1056.727458,342.543231,9.620590e-244,-0.348432,0.348432
98,saldo_medio_var5_hace2,3.00,0.00,1628.647567,377.343830,3.473307e-238,-0.346346,0.346346
100,saldo_medio_var5_ult1,3.00,0.00,1107.830596,335.149967,1.602023e-235,-0.343242,0.343242
101,saldo_medio_var5_ult3,2.76,0.00,1080.422776,282.659362,1.382146e-231,-0.343002,0.343002
99,saldo_medio_var5_hace3,1.05,0.00,923.548332,110.213467,7.277792e-224,-0.332773,0.332773
33,num_var35,3.00,0.00,3.344122,2.213098,1.888756e-181,-0.285209,0.285209
130,var38,107207.82,86219.97,117959.156918,99678.280590,9.853829e-54,-0.165152,0.165152


In [31]:
# p-value가 너무 작기에 여러 가설을 동시에 검정하면 다중비교 문제가 발생한다, 
# Benjamini-Hochberg 절차를 적용하여 다중검정으로 인한 FDR을 통제한다

import numpy as np

p_values = continuous_stats["p_value"].values

m = len(p_values)

order = np.argsort(p_values)
sorted_p = p_values[order]

adjusted = sorted_p * m / np.arange(1, m + 1)

# 뒤에서부터 누적 최솟값 적용
adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]

# 최대 1로 제한
adjusted = np.minimum(adjusted, 1)

# 원래 순서로 복원
p_adjusted = np.empty(m)
p_adjusted[order] = adjusted

continuous_stats["p_adjusted"] = p_adjusted

In [32]:
continuous_stats.sort_values(
    "abs_effect",
    ascending=False
).head(20)

,variable,median_target0,median_target1,mean_target0,mean_target1,p_value,rank_biserial,abs_effect,p_adjusted
1,var15,27.00,38.00,32.946406,39.680519,2.147072e-305,0.397280,0.397280,2.812664e-303
50,saldo_var30,3.00,0.00,14154.089498,2164.364721,5.672663e-286,-0.381658,0.381658,3.715594e-284
56,saldo_var42,3.00,0.00,7435.742356,1268.797769,5.370136e-245,-0.352087,0.352087,2.344959e-243
38,saldo_var5,3.00,0.00,1056.727458,342.543231,9.620590e-244,-0.348432,0.348432,3.150743e-242
98,saldo_medio_var5_hace2,3.00,0.00,1628.647567,377.343830,3.473307e-238,-0.346346,0.346346,9.100063e-237
100,saldo_medio_var5_ult1,3.00,0.00,1107.830596,335.149967,1.602023e-235,-0.343242,0.343242,3.497751e-234
101,saldo_medio_var5_ult3,2.76,0.00,1080.422776,282.659362,1.382146e-231,-0.343002,0.343002,2.586588e-230
99,saldo_medio_var5_hace3,1.05,0.00,923.548332,110.213467,7.277792e-224,-0.332773,0.332773,1.191738e-222
33,num_var35,3.00,0.00,3.344122,2.213098,1.888756e-181,-0.285209,0.285209,2.749190e-180
130,var38,107207.82,86219.97,117959.156918,99678.280590,9.853829e-54,-0.165152,0.165152,1.290852e-52


In [33]:
(continuous_stats["p_adjusted"] < 0.05).sum()

np.int64(62)

In [34]:
significant_continuous = continuous_stats[
    continuous_stats["p_adjusted"] < 0.05
].copy()

significant_continuous = significant_continuous.sort_values(
    "abs_effect",
    ascending=False
)

significant_continuous.head(20)

,variable,median_target0,median_target1,mean_target0,mean_target1,p_value,rank_biserial,abs_effect,p_adjusted
1,var15,27.00,38.00,32.946406,39.680519,2.147072e-305,0.397280,0.397280,2.812664e-303
50,saldo_var30,3.00,0.00,14154.089498,2164.364721,5.672663e-286,-0.381658,0.381658,3.715594e-284
56,saldo_var42,3.00,0.00,7435.742356,1268.797769,5.370136e-245,-0.352087,0.352087,2.344959e-243
38,saldo_var5,3.00,0.00,1056.727458,342.543231,9.620590e-244,-0.348432,0.348432,3.150743e-242
98,saldo_medio_var5_hace2,3.00,0.00,1628.647567,377.343830,3.473307e-238,-0.346346,0.346346,9.100063e-237
100,saldo_medio_var5_ult1,3.00,0.00,1107.830596,335.149967,1.602023e-235,-0.343242,0.343242,3.497751e-234
101,saldo_medio_var5_ult3,2.76,0.00,1080.422776,282.659362,1.382146e-231,-0.343002,0.343002,2.586588e-230
99,saldo_medio_var5_hace3,1.05,0.00,923.548332,110.213467,7.277792e-224,-0.332773,0.332773,1.191738e-222
33,num_var35,3.00,0.00,3.344122,2.213098,1.888756e-181,-0.285209,0.285209,2.749190e-180
130,var38,107207.82,86219.97,117959.156918,99678.280590,9.853829e-54,-0.165152,0.165152,1.290852e-52


In [35]:
# 상관관계 분석
corr = df_clean[continuous_cols].corr()

high_corr = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        
        value = corr.iloc[i, j]
        
        if abs(value) >= 0.8:
            high_corr.append({
                "variable_1": corr.columns[i],
                "variable_2": corr.columns[j],
                "correlation": value
            })

high_corr = pd.DataFrame(high_corr)

high_corr.sort_values(
    "correlation",
    key=abs,
    ascending=False
).head(30)

,variable_1,variable_2,correlation
56,num_var37_0,num_var37,1.000000
17,imp_op_var41_efect_ult3,imp_op_var39_efect_ult3,0.999271
117,num_op_var41_efect_ult3,num_op_var39_efect_ult3,0.999244
114,num_op_var41_efect_ult1,num_op_var39_efect_ult1,0.998966
90,saldo_var33,saldo_medio_var33_ult1,0.998626
13,imp_op_var41_efect_ult1,imp_op_var39_efect_ult1,0.998378
98,imp_aport_var17_hace3,saldo_medio_var17_hace3,0.998288
78,saldo_var17,saldo_medio_var17_ult3,0.997857
105,num_med_var45_ult3,num_var45_ult3,0.997806
77,saldo_var17,saldo_medio_var17_ult1,0.997789


In [36]:
# 동일한 변수 찾기
exact_duplicates = high_corr[
    high_corr["correlation"].abs() >= 0.999999
]

exact_duplicates

,variable_1,variable_2,correlation
56,num_var37_0,num_var37,1.0


In [37]:
# TARGET과의 연관성 확인
effect_map = continuous_stats.set_index( "variable" )["abs_effect"] 
high_corr["effect_1"] = ( high_corr["variable_1"].map(effect_map) ) 
high_corr["effect_2"] = ( high_corr["variable_2"].map(effect_map) ) 
high_corr["effect_diff"] = ( high_corr["effect_1"] - high_corr["effect_2"] ).abs()

In [38]:
high_corr.sort_values(
    "correlation",
    key=abs,
    ascending=False
).head(30)

,variable_1,variable_2,correlation,effect_1,effect_2,effect_diff
56,num_var37_0,num_var37,1.000000,0.004015,0.004015,0.000000e+00
17,imp_op_var41_efect_ult3,imp_op_var39_efect_ult3,0.999271,0.022676,0.022527,1.484381e-04
117,num_op_var41_efect_ult3,num_op_var39_efect_ult3,0.999244,0.020564,0.020479,8.506963e-05
114,num_op_var41_efect_ult1,num_op_var39_efect_ult1,0.998966,0.017372,0.017612,2.402103e-04
90,saldo_var33,saldo_medio_var33_ult1,0.998626,0.000657,0.000657,0.000000e+00
13,imp_op_var41_efect_ult1,imp_op_var39_efect_ult1,0.998378,0.018871,0.019088,2.164829e-04
98,imp_aport_var17_hace3,saldo_medio_var17_hace3,0.998288,0.000301,0.000233,6.848189e-05
78,saldo_var17,saldo_medio_var17_ult3,0.997857,0.000814,0.000924,1.094344e-04
105,num_med_var45_ult3,num_var45_ult3,0.997806,0.047917,0.052925,5.008053e-03
77,saldo_var17,saldo_medio_var17_ult1,0.997789,0.000814,0.000924,1.094982e-04


In [39]:
continuous_stats[
    continuous_stats["variable"].isin(
        ["num_var37_0", "num_var37"]
    )
]

,variable,median_target0,median_target1,mean_target0,mean_target1,p_value,rank_biserial,abs_effect,p_adjusted
35,num_var37_0,0.0,0.0,0.418041,0.436835,0.382579,0.004015,0.004015,0.455616
36,num_var37,0.0,0.0,0.418041,0.436835,0.382579,0.004015,0.004015,0.455616


In [40]:
df_clean.drop(columns=["num_var37_0"])

,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,imp_op_var40_ult1,...,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET,var3_missing
0,2.0,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0,0
1,2.0,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0,0
2,2.0,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0,0
3,2.0,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0,0
4,2.0,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76015,2.0,48,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60926.490000,0,0
76016,2.0,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,118634.520000,0,0
76017,2.0,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,74028.150000,0,0
76018,2.0,25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,84278.160000,0,0


In [41]:
high_corr_95 = high_corr[
    high_corr["correlation"].abs() >= 0.95
].copy()

print("상관계수 |r| >= 0.95인 변수쌍:", len(high_corr_95))

high_corr_95.head(30)

상관계수 |r| >= 0.95인 변수쌍: 57


,variable_1,variable_2,correlation,effect_1,effect_2,effect_diff
1,imp_op_var39_comer_ult1,imp_op_var41_comer_ult1,0.961781,0.009688,0.008069,1.618376e-03
4,imp_op_var39_comer_ult3,imp_op_var41_comer_ult3,0.959840,0.010043,0.007986,2.057353e-03
13,imp_op_var41_efect_ult1,imp_op_var39_efect_ult1,0.998378,0.018871,0.019088,2.164829e-04
17,imp_op_var41_efect_ult3,imp_op_var39_efect_ult3,0.999271,0.022676,0.022527,1.484381e-04
19,imp_op_var41_ult1,imp_op_var39_ult1,0.991103,0.002255,0.002687,4.323147e-04
24,num_op_var40_ult1,num_op_var40_ult3,0.955964,0.001364,0.001104,2.608960e-04
26,num_op_var41_hace2,num_op_var39_hace2,0.992596,0.000912,0.001862,9.495898e-04
29,num_op_var41_hace3,num_op_var39_hace3,0.989116,0.001928,0.002024,9.582912e-05
31,num_op_var41_ult1,num_op_var39_ult1,0.986411,0.004464,0.004956,4.921635e-04
39,num_op_var41_ult3,num_op_var39_ult3,0.988861,0.005948,0.006762,8.134456e-04


In [42]:
import pandas as pd
import numpy as np

# --------------------------------
# 1. 상관관계 기준
# --------------------------------

threshold = 0.95

corr = df_clean[continuous_cols].corr()

# --------------------------------
# 2. Union-Find
# --------------------------------

parent = {col: col for col in continuous_cols}


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(x, y):
    root_x = find(x)
    root_y = find(y)

    if root_x != root_y:
        parent[root_y] = root_x


# --------------------------------
# 3. 높은 상관관계 변수들을 그룹화
# --------------------------------

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):

        value = corr.iloc[i, j]

        if abs(value) >= threshold:
            union(
                corr.columns[i],
                corr.columns[j]
            )


# --------------------------------
# 4. 그룹 생성
# --------------------------------

groups = {}

for col in continuous_cols:

    root = find(col)

    if root not in groups:
        groups[root] = []

    groups[root].append(col)


# --------------------------------
# 5. 효과크기 정보 연결
# --------------------------------

effect_map = (
    continuous_stats
    .set_index("variable")["abs_effect"]
)


# --------------------------------
# 6. 각 그룹의 대표 변수 선정
# --------------------------------

representative_features = []

for group in groups.values():

    if len(group) == 1:
        representative_features.append(group[0])

    else:

        group_effect = (
            effect_map
            .reindex(group)
            .fillna(0)
        )

        representative = group_effect.idxmax()

        representative_features.append(
            representative
        )


print("전체 연속형 변수:", len(continuous_cols))
print("중복 그룹 수:", sum(len(g) > 1 for g in groups.values()))
print("대표 변수 수:", len(representative_features))

전체 연속형 변수: 131
중복 그룹 수: 24
대표 변수 수: 97


In [43]:
correlated_groups = [
    group
    for group in groups.values()
    if len(group) > 1
]

for i, group in enumerate(correlated_groups, 1):

    group_effect = (
        effect_map
        .reindex(group)
        .sort_values(ascending=False)
    )

    print(f"\n===== Group {i} =====")

    for variable, effect in group_effect.items():

        print(
            f"{variable:<40} "
            f"effect={effect:.4f}"
        )


===== Group 1 =====
imp_op_var39_comer_ult1                  effect=0.0097
imp_op_var41_comer_ult1                  effect=0.0081

===== Group 2 =====
imp_op_var39_comer_ult3                  effect=0.0100
imp_op_var41_comer_ult3                  effect=0.0080

===== Group 3 =====
imp_op_var39_efect_ult1                  effect=0.0191
imp_op_var41_efect_ult1                  effect=0.0189

===== Group 4 =====
imp_op_var41_efect_ult3                  effect=0.0227
imp_op_var39_efect_ult3                  effect=0.0225

===== Group 5 =====
imp_op_var39_ult1                        effect=0.0027
imp_op_var41_ult1                        effect=0.0023

===== Group 6 =====
num_op_var40_ult1                        effect=0.0014
num_op_var40_ult3                        effect=0.0011

===== Group 7 =====
num_op_var39_hace2                       effect=0.0019
num_op_var41_hace2                       effect=0.0009

===== Group 8 =====
num_op_var39_hace3                       effect=0.0020
num_op_

In [44]:
# ==========================================
# 9. Train / Validation / Test 분리
# ==========================================

from sklearn.model_selection import train_test_split

# Target
y = df_clean["TARGET"]

# ID와 Target 제외
X_raw = df_clean.drop(
    columns=["TARGET", "ID"],
    errors="ignore"
)

# Train 60% / 나머지 40%
X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw,
    y,
    test_size=0.4,
    stratify=y,
    random_state=42
)

# Validation 20% / Test 20%
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("===== 데이터 크기 =====")
print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)
print("X_test  :", X_test.shape)

print("\ny_train :", y_train.shape)
print("y_valid :", y_valid.shape)
print("y_test  :", y_test.shape)

print("\n===== TARGET 비율 =====")

print("\nTrain")
print(y_train.value_counts(normalize=True))

print("\nValidation")
print(y_valid.value_counts(normalize=True))

print("\nTest")
print(y_test.value_counts(normalize=True))

===== 데이터 크기 =====
X_train : (45612, 336)
X_valid : (15204, 336)
X_test  : (15204, 336)

y_train : (45612,)
y_valid : (15204,)
y_test  : (15204,)

===== TARGET 비율 =====

Train
TARGET
0    0.960427
1    0.039573
Name: proportion, dtype: float64

Validation
TARGET
0    0.960405
1    0.039595
Name: proportion, dtype: float64

Test
TARGET
0    0.960471
1    0.039529
Name: proportion, dtype: float64


In [45]:
# ==========================================
# 9-1. Train 기준 전처리 상태 확인
# ==========================================

import numpy as np
import pandas as pd

# Train의 결측치 확인
train_missing = X_train.isna().sum()

print("===== Train 결측치 =====")
print(train_missing[train_missing > 0])

# Validation / Test 결측치 개수
print("\n===== 결측치 총 개수 =====")
print("Train:", X_train.isna().sum().sum())
print("Valid:", X_valid.isna().sum().sum())
print("Test :", X_test.isna().sum().sum())

# 변수 타입 확인
print("\n===== 변수 타입 =====")
print(X_train.dtypes.value_counts())

# 상수 변수 확인
constant_features = [
    col
    for col in X_train.columns
    if X_train[col].nunique(dropna=False) <= 1
]

print("\n===== Train 기준 상수 변수 =====")
print("개수:", len(constant_features))
print(constant_features)

===== Train 결측치 =====
Series([], dtype: int64)

===== 결측치 총 개수 =====
Train: 0
Valid: 0
Test : 0

===== 변수 타입 =====
int64      224
float64    112
Name: count, dtype: int64

===== Train 기준 상수 변수 =====
개수: 23
['ind_var13_medio_0', 'ind_var13_medio', 'ind_var34_0', 'ind_var34', 'num_var13_medio_0', 'num_var13_medio', 'num_var34_0', 'num_var34', 'saldo_var13_medio', 'saldo_var34', 'delta_imp_amort_var34_1y3', 'delta_imp_reemb_var33_1y3', 'delta_num_reemb_var33_1y3', 'imp_amort_var34_ult1', 'imp_reemb_var17_hace3', 'imp_reemb_var33_ult1', 'num_meses_var13_medio_ult3', 'num_reemb_var17_hace3', 'num_reemb_var33_ult1', 'saldo_medio_var13_medio_hace2', 'saldo_medio_var13_medio_ult1', 'saldo_medio_var13_medio_ult3', 'saldo_medio_var29_hace3']


In [46]:
# ==========================================
# 9-2. Train 기준 상수 변수 제거
# ==========================================

# Train에서 확인된 상수 변수 제거
X_train_clean = X_train.drop(
    columns=constant_features
)

X_valid_clean = X_valid.drop(
    columns=constant_features
)

X_test_clean = X_test.drop(
    columns=constant_features
)

print("===== 상수 변수 제거 후 =====")
print("X_train:", X_train_clean.shape)
print("X_valid:", X_valid_clean.shape)
print("X_test :", X_test_clean.shape)

print("\n제거된 변수 개수:", len(constant_features))
print("남은 변수 개수:", X_train_clean.shape[1])

===== 상수 변수 제거 후 =====
X_train: (45612, 313)
X_valid: (15204, 313)
X_test : (15204, 313)

제거된 변수 개수: 23
남은 변수 개수: 313


In [47]:
# ==========================================
# 8-1. 통계적 변수 선택 - Train 기준
# FDR 보정: Benjamini-Hochberg
# ==========================================

from scipy.stats import mannwhitneyu
import pandas as pd
import numpy as np


# ------------------------------------------
# 1. Mann-Whitney U 검정
# ------------------------------------------

stat_results = []

for col in X_train_clean.columns:

    group0 = X_train_clean.loc[y_train == 0, col]
    group1 = X_train_clean.loc[y_train == 1, col]

    try:
        stat, p_value = mannwhitneyu(
            group0,
            group1,
            alternative="two-sided"
        )

        n0 = len(group0)
        n1 = len(group1)

        # Rank-biserial correlation
        effect = 1 - (2 * stat) / (n0 * n1)

    except Exception:
        p_value = np.nan
        effect = np.nan

    stat_results.append({
        "variable": col,
        "p_value": p_value,
        "effect": effect,
        "abs_effect": abs(effect)
        if pd.notna(effect) else np.nan
    })


continuous_stats = pd.DataFrame(stat_results)


# ------------------------------------------
# 2. Benjamini-Hochberg FDR 보정
# ------------------------------------------

def benjamini_hochberg(p_values):

    p_values = np.asarray(p_values, dtype=float)

    adjusted = np.full_like(
        p_values,
        np.nan,
        dtype=float
    )

    valid_idx = np.where(
        ~np.isnan(p_values)
    )[0]

    p_valid = p_values[valid_idx]

    m = len(p_valid)

    # p-value 오름차순 정렬
    order = np.argsort(p_valid)

    sorted_p = p_valid[order]

    # BH adjusted p-value
    adjusted_sorted = (
        sorted_p * m /
        np.arange(1, m + 1)
    )

    # 뒤에서부터 누적 최소값
    adjusted_sorted = np.minimum.accumulate(
        adjusted_sorted[::-1]
    )[::-1]

    # 1보다 클 수 없음
    adjusted_sorted = np.minimum(
        adjusted_sorted,
        1.0
    )

    # 원래 순서로 복원
    adjusted_valid = np.empty_like(
        adjusted_sorted
    )

    adjusted_valid[order] = adjusted_sorted

    adjusted[valid_idx] = adjusted_valid

    return adjusted


continuous_stats["p_adjusted"] = (
    benjamini_hochberg(
        continuous_stats["p_value"]
    )
)


# ------------------------------------------
# 3. FDR 유의 여부
# ------------------------------------------

continuous_stats["significant"] = (
    continuous_stats["p_adjusted"] < 0.05
)


# ------------------------------------------
# 4. 효과크기 기준 정렬
# ------------------------------------------

continuous_stats = (
    continuous_stats
    .sort_values(
        ["significant", "abs_effect"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------
# 5. 결과 확인
# ------------------------------------------

print("===== 통계적 변수 선택 결과 =====")

print(
    "전체 변수:",
    len(continuous_stats)
)

print(
    "FDR 유의 변수:",
    continuous_stats["significant"].sum()
)

print("\n===== 상위 변수 =====")

display(
    continuous_stats[
        [
            "variable",
            "p_value",
            "p_adjusted",
            "effect",
            "abs_effect",
            "significant"
        ]
    ].head(30)
)

===== 통계적 변수 선택 결과 =====
전체 변수: 313
FDR 유의 변수: 132

===== 상위 변수 =====


,variable,p_value,p_adjusted,effect,abs_effect,significant
0,var15,4.376503e-187,3.424614e-185,0.400563,0.400563,True
1,num_meses_var5_ult3,5.376273e-204,8.413867e-202,-0.378243,0.378243,True
2,saldo_var30,1.729939e-158,6.768385e-157,-0.365743,0.365743,True
3,num_var30,4.999099e-199,5.215727e-197,-0.342330,0.342330,True
4,saldo_var5,3.835850e-141,1.334023e-139,-0.341239,0.341239,True
5,saldo_var42,3.836772e-135,1.200910e-133,-0.336441,0.336441,True
6,saldo_medio_var5_hace2,2.567534e-134,7.305801e-133,-0.334608,0.334608,True
7,saldo_medio_var5_ult3,3.071573e-132,6.867160e-131,-0.333427,0.333427,True
8,saldo_medio_var5_ult1,5.347190e-134,1.394726e-132,-0.333098,0.333098,True
9,ind_var30,4.031131e-214,1.261744e-211,-0.331966,0.331966,True


In [48]:
# ==========================================
# 8-1-2. Train 기준 상관관계 분석
# ==========================================

import numpy as np
import pandas as pd

# 통계적으로 유의한 변수만 후보로 사용
stat_candidates = continuous_stats.loc[
    continuous_stats["significant"] == True,
    "variable"
].tolist()

print("FDR 유의 변수:", len(stat_candidates))


# ------------------------------------------
# 변수 간 상관관계 계산
# ------------------------------------------

corr_matrix = X_train_clean[stat_candidates].corr(
    method="spearman"
)

# 높은 상관관계 쌍 추출
correlation_threshold = 0.95

high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):

        corr = corr_matrix.iloc[i, j]

        if abs(corr) >= correlation_threshold:

            high_corr_pairs.append({
                "variable_1": corr_matrix.columns[i],
                "variable_2": corr_matrix.columns[j],
                "correlation": corr
            })

high_corr = pd.DataFrame(high_corr_pairs)

print("\n===== 높은 상관관계 변수 쌍 =====")
print(
    "기준 |rho| >=",
    correlation_threshold
)

print(
    "변수 쌍 개수:",
    len(high_corr)
)

display(
    high_corr
    .sort_values(
        "correlation",
        key=lambda x: x.abs(),
        ascending=False
    )
    .head(30)
)

FDR 유의 변수: 132

===== 높은 상관관계 변수 쌍 =====
기준 |rho| >= 0.95
변수 쌍 개수: 193


,variable_1,variable_2,correlation
5,ind_var8_0,num_var8_0,1.000000
180,num_var40,num_var39,1.000000
191,ind_var20_0,num_var20_0,1.000000
189,ind_var1,num_var1,1.000000
142,ind_var25_0,ind_var25,1.000000
130,num_var25_0,num_var25,1.000000
170,ind_var40,num_var40,1.000000
171,ind_var40,num_var39,1.000000
176,ind_var39,num_var39,1.000000
175,ind_var39,num_var40,1.000000


In [49]:
# 통계적으로 유의한 변수 중 상당수가 서로 중복 정보를 가지고 있어다. 
# 따라서 132개를 그대로 쓰는 것보다 대표 변수를 선정하겠다.

In [50]:
# ==========================================
# 8-1-3. 고상관 변수 그룹 생성 및 대표 변수 선정
# ==========================================

# 통계적 후보 변수 전체
candidate_features = stat_candidates.copy()

# ------------------------------------------
# 1. Union-Find로 상관관계 그룹 생성
# ------------------------------------------

parent = {
    feature: feature
    for feature in candidate_features
}


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(x, y):

    root_x = find(x)
    root_y = find(y)

    if root_x != root_y:
        parent[root_y] = root_x


# 높은 상관관계 변수들을 같은 그룹으로 연결
for _, row in high_corr.iterrows():

    var1 = row["variable_1"]
    var2 = row["variable_2"]

    union(var1, var2)


# ------------------------------------------
# 2. 최종 그룹 생성
# ------------------------------------------

groups = {}

for feature in candidate_features:

    root = find(feature)

    if root not in groups:
        groups[root] = []

    groups[root].append(feature)


# ------------------------------------------
# 3. 각 그룹에서 효과크기 최대 변수 선택
# ------------------------------------------

effect_map = continuous_stats.set_index(
    "variable"
)["abs_effect"]


representative_features = []

group_summary = []


for group_id, group in enumerate(
    groups.values(),
    start=1
):

    # 효과크기 기준 정렬
    group_effect = (
        effect_map
        .reindex(group)
        .sort_values(
            ascending=False
        )
    )

    representative = group_effect.index[0]

    representative_features.append(
        representative
    )

    group_summary.append({
        "group": group_id,
        "size": len(group),
        "representative": representative,
        "representative_effect":
            group_effect.iloc[0],
        "variables": ", ".join(group)
    })


group_summary = pd.DataFrame(
    group_summary
)


# ------------------------------------------
# 4. 결과 확인
# ------------------------------------------

print("===== 대표 변수 선정 결과 =====")

print(
    "통계적 후보 변수:",
    len(candidate_features)
)

print(
    "상관관계 그룹:",
    len(groups)
)

print(
    "대표 변수:",
    len(representative_features)
)

print(
    "제거된 중복 변수:",
    len(candidate_features)
    - len(representative_features)
)

print("\n===== 그룹별 대표 변수 =====")

display(
    group_summary
    .sort_values(
        "size",
        ascending=False
    )
    .head(30)
)

===== 대표 변수 선정 결과 =====
통계적 후보 변수: 132
상관관계 그룹: 60
대표 변수: 60
제거된 중복 변수: 72

===== 그룹별 대표 변수 =====


,group,size,representative,representative_effect,variables
37,38,10,num_var26_0,0.021170,"num_var26_0, num_var26, ind_var26_0, ind_var26..."
25,26,9,saldo_var13_corto,0.034137,"saldo_var13_corto, saldo_medio_var13_corto_ult..."
56,57,8,saldo_var40,0.003785,"saldo_var40, ind_var40, ind_var39, num_var40, ..."
27,28,6,saldo_var12,0.032739,"saldo_var12, num_var12, ind_var12, saldo_medio..."
28,29,6,num_meses_var8_ult3,0.030689,"num_meses_var8_ult3, ind_var8, num_var8, saldo..."
49,50,5,num_var13_largo_0,0.010079,"num_var13_largo_0, ind_var13_largo_0, saldo_va..."
20,21,5,saldo_var13,0.043844,"saldo_var13, ind_var13, num_var13, num_var13_0..."
35,36,4,imp_op_var41_efect_ult1,0.025264,"imp_op_var41_efect_ult1, imp_op_var39_efect_ul..."
31,32,4,imp_op_var41_efect_ult3,0.028428,"imp_op_var41_efect_ult3, imp_op_var39_efect_ul..."
30,31,3,saldo_var24,0.028592,"saldo_var24, num_var24, ind_var24"


In [51]:
# 통계적 기준에서 132 → 60개로 압축

In [52]:
# ==========================================
# 8-2. LASSO Logistic Regression 변수 선택
# ==========================================

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

# ------------------------------------------
# 1. Train 데이터 표준화
# ------------------------------------------

scaler_lasso = StandardScaler()

X_train_scaled = scaler_lasso.fit_transform(
    X_train_clean
)

# ------------------------------------------
# 2. L1 Logistic Regression
# ------------------------------------------

lasso_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.1,
    max_iter=1000,
    random_state=42
)

lasso_model.fit(
    X_train_scaled,
    y_train
)

# ------------------------------------------
# 3. 계수 확인
# ------------------------------------------

lasso_coef = pd.Series(
    lasso_model.coef_[0],
    index=X_train_clean.columns
)

# 0이 아닌 변수 = LASSO 선택 변수
lasso_selected = lasso_coef[
    lasso_coef != 0
].index.tolist()

print("===== LASSO 변수 선택 결과 =====")
print("전체 변수:", len(X_train_clean.columns))
print("선택 변수:", len(lasso_selected))
print("제거 변수:", len(X_train_clean.columns) - len(lasso_selected))

# ------------------------------------------
# 4. 선택된 변수 확인
# ------------------------------------------

lasso_result = (
    lasso_coef
    .loc[lasso_selected]
    .sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )
    .to_frame("coefficient")
)

display(lasso_result.head(30))

c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


===== LASSO 변수 선택 결과 =====
전체 변수: 313
선택 변수: 109
제거 변수: 204


,coefficient
var15,0.492672
var38,-0.477429
num_meses_var5_ult3,-0.469501
ind_var13,-0.406579
saldo_medio_var8_hace2,-0.338477
ind_var30,-0.288832
num_var22_ult3,0.164746
saldo_medio_var5_hace3,-0.152342
ind_var30_0,0.151723
saldo_var5,-0.132118


In [53]:
# ==========================================
# 8-3. Tree 기반 변수 선택
# Random Forest Feature Importance
# ==========================================

from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

# ------------------------------------------
# 1. Random Forest 학습
# ------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_clean,
    y_train
)

# ------------------------------------------
# 2. Feature Importance
# ------------------------------------------

rf_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train_clean.columns
).sort_values(
    ascending=False
)

print("===== Random Forest Feature Importance =====")

display(
    rf_importance
    .head(30)
    .to_frame("importance")
)

# ------------------------------------------
# 3. 중요도 기준 변수 선택
# ------------------------------------------

# 상위 20%
rf_threshold = rf_importance.quantile(0.80)

rf_selected = rf_importance[
    rf_importance >= rf_threshold
].index.tolist()

print("\n===== Random Forest 변수 선택 =====")
print("전체 변수:", len(X_train_clean.columns))
print("선택 변수:", len(rf_selected))
print("중요도 기준:", rf_threshold)

===== Random Forest Feature Importance =====


,importance
var15,0.158532
saldo_var30,0.071476
saldo_var5,0.066157
saldo_var42,0.047282
saldo_medio_var5_ult3,0.045476
saldo_medio_var5_hace2,0.040095
num_meses_var5_ult3,0.034261
saldo_medio_var5_ult1,0.032863
num_var42,0.029494
saldo_medio_var5_hace3,0.025622



===== Random Forest 변수 선택 =====
전체 변수: 313
선택 변수: 63
중요도 기준: 0.002007463961849342


In [54]:
# ==========================================
# 8-3-2. XGBoost Feature Importance
# ==========================================

from xgboost import XGBClassifier
import pandas as pd
import numpy as np

# ------------------------------------------
# 1. XGBoost 학습
# ------------------------------------------

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_clean,
    y_train
)

# ------------------------------------------
# 2. Feature Importance
# ------------------------------------------

xgb_importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_train_clean.columns
).sort_values(
    ascending=False
)

print("===== XGBoost Feature Importance =====")

display(
    xgb_importance
    .head(30)
    .to_frame("importance")
)

# ------------------------------------------
# 3. 중요도 기준 변수 선택
# ------------------------------------------

xgb_threshold = xgb_importance.quantile(0.80)

xgb_selected = xgb_importance[
    xgb_importance >= xgb_threshold
].index.tolist()

print("\n===== XGBoost 변수 선택 =====")
print("전체 변수:", len(X_train_clean.columns))
print("선택 변수:", len(xgb_selected))
print("중요도 기준:", xgb_threshold)

===== XGBoost Feature Importance =====


,importance
ind_var30,0.123872
num_var4,0.051711
saldo_var30,0.037237
ind_var26_0,0.031315
ind_var13_0,0.027596
var15,0.026210
ind_var30_0,0.017943
num_var35,0.016867
num_var5,0.016434
ind_var13,0.014192



===== XGBoost 변수 선택 =====
전체 변수: 313
선택 변수: 63
중요도 기준: 0.004943007417023182


In [55]:
# ==========================================
# 8-4. 변수 선택 방법 간 공통 변수 비교
# ==========================================

# 각 방법별 변수 집합
stat_set = set(stat_candidates)
lasso_set = set(lasso_selected)
rf_set = set(rf_selected)
xgb_set = set(xgb_selected)

# ------------------------------------------
# 1. 방법별 선택 변수 수
# ------------------------------------------

print("===== 변수 선택 결과 =====")
print("통계검정:", len(stat_set))
print("LASSO:", len(lasso_set))
print("Random Forest:", len(rf_set))
print("XGBoost:", len(xgb_set))


# ------------------------------------------
# 2. 전체 방법에서 공통으로 선택된 변수
# ------------------------------------------

all_common = (
    stat_set
    & lasso_set
    & rf_set
    & xgb_set
)

print("\n===== 4개 방법 공통 변수 =====")
print("개수:", len(all_common))
print(sorted(all_common))


# ------------------------------------------
# 3. 통계 + LASSO + Tree 공통
# ------------------------------------------

stat_lasso_tree = (
    stat_set
    & lasso_set
    & rf_set
    & xgb_set
)

print("\n4개 방법 공통 변수 개수:",
      len(stat_lasso_tree))


# ------------------------------------------
# 4. 각 변수의 선택 방법 개수
# ------------------------------------------

all_selected = (
    stat_set
    | lasso_set
    | rf_set
    | xgb_set
)

method_count = []

for variable in all_selected:

    count = sum([
        variable in stat_set,
        variable in lasso_set,
        variable in rf_set,
        variable in xgb_set
    ])

    method_count.append({
        "variable": variable,
        "selection_count": count,
        "statistical": variable in stat_set,
        "lasso": variable in lasso_set,
        "random_forest": variable in rf_set,
        "xgboost": variable in xgb_set
    })

method_comparison = pd.DataFrame(
    method_count
).sort_values(
    ["selection_count", "variable"],
    ascending=[False, True]
)

print("\n===== 변수별 선택 방법 수 =====")

display(
    method_comparison.head(50)
)

===== 변수 선택 결과 =====
통계검정: 132
LASSO: 109
Random Forest: 63
XGBoost: 63

===== 4개 방법 공통 변수 =====
개수: 15
['ind_var13', 'ind_var30', 'num_meses_var5_ult3', 'num_var22_ult1', 'num_var35', 'num_var42_0', 'num_var45_ult1', 'saldo_medio_var5_hace2', 'saldo_medio_var5_hace3', 'saldo_medio_var5_ult1', 'saldo_var30', 'saldo_var42', 'saldo_var5', 'var15', 'var38']

4개 방법 공통 변수 개수: 15

===== 변수별 선택 방법 수 =====


,variable,selection_count,statistical,lasso,random_forest,xgboost
75,ind_var13,4,True,True,True,True
149,ind_var30,4,True,True,True,True
87,num_meses_var5_ult3,4,True,True,True,True
164,num_var22_ult1,4,True,True,True,True
23,num_var35,4,True,True,True,True
71,num_var42_0,4,True,True,True,True
110,num_var45_ult1,4,True,True,True,True
188,saldo_medio_var5_hace2,4,True,True,True,True
65,saldo_medio_var5_hace3,4,True,True,True,True
73,saldo_medio_var5_ult1,4,True,True,True,True
